<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## Kraken

Manages the Kraken data download and processing using the [ccxt](https://github.com/ccxt/ccxt) package.
Candles (OHLCV) are downloaded through `ccxt.kraken` and saved, one file per trading pair, into a
folder named after the exchange (e.g. `../data/kraken`).

Notes:

- Kraken's public OHLC endpoint only serves roughly the most recent **720 candles per timeframe**.
  For hourly data that is about 30 days of history. Longer backfills are therefore not possible
  through this API; keep the files updated regularly instead.
- Rate limits are handled by ccxt (`enableRateLimit=True`).

** Finally, datetime columns are in UTC. **

In [0]:
#| echo: false
#| output: asis
show_doc(retry_fetch_ohlcv)

---

### retry_fetch_ohlcv

```python
def retry_fetch_ohlcv(
    exchange, max_retries, symbol, timeframe, since, limit, verbose:bool=False
):
```

*Fetch a single page of OHLCV candles from an exchange, retrying on failure.*

Args:
    exchange (ccxt.Exchange): Instantiated ccxt exchange object
    max_retries (int): Maximum number of retries before raising the last error
    symbol (str): Trading pair symbol (e.g. "BTC/USD")
    timeframe (str): Candle timeframe (e.g. "1m", "1h", "1d")
    since (int): Start time in milliseconds since epoch (UTC)
    limit (int): Maximum number of candles to fetch
    verbose (bool, optional): If True, prints progress messages. Defaults to False

Returns:
    list: List of OHLCV candles [timestamp_ms, open, high, low, close, volume]

Raises:
    Exception: The last ccxt error if all retries fail

In [0]:
#| echo: false
#| output: asis
show_doc(scrape_ohlcv)

---

### scrape_ohlcv

```python
def scrape_ohlcv(
    exchange, symbol, timeframe, since, end:NoneType=None, max_retries:int=3, limit:int=720, verbose:bool=False
):
```

*Download OHLCV candles in pages of `limit` bars between `since` and `end`.*

Args:
    exchange (ccxt.Exchange): Instantiated ccxt exchange object (markets loaded)
    symbol (str): Trading pair symbol (e.g. "BTC/USD")
    timeframe (str): Candle timeframe (e.g. "1m", "1h", "1d")
    since (int or str): Start time in milliseconds since epoch or ISO 8601 string
    end (int or str, optional): End time in milliseconds or ISO 8601 string.
        Defaults to now.
    max_retries (int, optional): Retries per page. Defaults to 3
    limit (int, optional): Candles per request. Defaults to 720 (Kraken maximum)
    verbose (bool, optional): If True, prints progress messages. Defaults to False

Returns:
    list: List of OHLCV candles [timestamp_ms, open, high, low, close, volume]

In [0]:
#| echo: false
#| output: asis
show_doc(ohlcv_to_df)

---

### ohlcv_to_df

```python
def ohlcv_to_df(
    ohlcv, symbol
):
```

*Convert a raw ccxt OHLCV list into a tidy DataFrame.*

Args:
    ohlcv (list): List of candles [timestamp_ms, open, high, low, close, volume]
    symbol (str): Trading pair symbol added as the `pair` column

Returns:
    pandas.DataFrame: DataFrame with columns:
        - datetime: Candle timestamp (UTC, timezone-aware)
        - open, high, low, close, volume: OHLCV values
        - pair: Trading pair symbol
    Sorted by datetime with duplicate timestamps removed.

In [0]:
#| echo: false
#| output: asis
show_doc(kraken_ohlcv)

---

### kraken_ohlcv

```python
def kraken_ohlcv(
    symbol:str='BTC/USD', timeframe:str='1h', since:NoneType=None, end:NoneType=None, max_retries:int=3,
    limit:int=720, exchange:NoneType=None, verbose:bool=False
):
```

*Download OHLCV candles for a symbol from Kraken.*

Args:
    symbol (str, optional): Trading pair symbol. Defaults to "BTC/USD"
    timeframe (str, optional): Candle timeframe. Defaults to "1h"
    since (int or str, optional): Start time (ms since epoch or ISO 8601).
        If None, downloads the most recent ~`limit` candles (Kraken only
        serves about 720 recent candles per timeframe).
    end (int or str, optional): End time (ms or ISO 8601). Defaults to now
    max_retries (int, optional): Retries per page. Defaults to 3
    limit (int, optional): Candles per request. Defaults to 720
    exchange (ccxt.kraken, optional): Reusable exchange instance. If None, a new one is created
    verbose (bool, optional): If True, prints progress messages. Defaults to False

Returns:
    pandas.DataFrame: Tidy OHLCV DataFrame (see `ohlcv_to_df`)

#### Example / tests

In [ ]:
#|eval: false
# Quick live test: download ~2 days of hourly BTC/USD candles
start = (pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=2)).strftime('%Y-%m-%dT%H:%M:%SZ')
df_test = kraken_ohlcv(symbol='BTC/USD', timeframe='1h', since=start)
assert not df_test.empty
assert list(df_test.columns) == ['datetime', 'open', 'high', 'low', 'close', 'volume', 'pair']
assert df_test['datetime'].dt.tz is not None
assert df_test['datetime'].is_monotonic_increasing
assert (df_test['pair'] == 'BTC/USD').all()
assert len(df_test) > 24
df_test.tail()

,datetime,open,high,low,close,volume,pair
43,2026-07-08 08:00:00+00:00,62838.9,62876.6,61821.8,61954.2,283.793404,BTC/USD
44,2026-07-08 09:00:00+00:00,61956.2,62097.4,61700.0,62005.7,45.414531,BTC/USD
45,2026-07-08 10:00:00+00:00,62005.7,62125.3,61840.1,62105.4,59.441858,BTC/USD
46,2026-07-08 11:00:00+00:00,62125.3,62338.9,61991.7,62247.0,77.719395,BTC/USD
47,2026-07-08 12:00:00+00:00,62247.0,62358.2,62013.0,62084.9,55.661273,BTC/USD


In [ ]:
#|eval: false
# Offline test: ohlcv_to_df shapes raw candles correctly and removes duplicates
sample = [[1700000000000, 1.0, 2.0, 0.5, 1.5, 10.0],
          [1700003600000, 1.5, 2.5, 1.0, 2.0, 20.0],
          [1700003600000, 1.5, 2.5, 1.0, 2.0, 20.0]]  # duplicate on purpose
df_sample = ohlcv_to_df(sample, 'TEST/USD')
assert len(df_sample) == 2
assert list(df_sample.columns) == ['datetime', 'open', 'high', 'low', 'close', 'volume', 'pair']
assert str(df_sample['datetime'].dt.tz) == 'UTC'
df_sample

,datetime,open,high,low,close,volume,pair
0,2023-11-14 22:13:20+00:00,1.0,2.0,0.5,1.5,10.0,TEST/USD
1,2023-11-14 23:13:20+00:00,1.5,2.5,1.0,2.0,20.0,TEST/USD


In [0]:
#| echo: false
#| output: asis
show_doc(kraken_usd_tokens)

---

### kraken_usd_tokens

```python
def kraken_usd_tokens(
    exchange:NoneType=None
):
```

*Retrieves all active Kraken spot trading pairs quoted in USD.*

Args:
    exchange (ccxt.kraken, optional): Reusable exchange instance. If None, a new one is created

Returns:
    pandas.DataFrame: DataFrame with columns:
        - id: Kraken market id (e.g. 'XBTUSD')
        - symbol: Unified ccxt symbol (e.g. 'BTC/USD')
        - base: Base currency (e.g. 'BTC')
        - quote: Always 'USD' for this filtered dataset
        - active: Whether the market is currently active

In [ ]:
#|eval: false
# Live test: token universe contains the major pairs
tokens = kraken_usd_tokens()
assert not tokens.empty
assert 'BTC/USD' in tokens['symbol'].tolist()
assert (tokens['quote'] == 'USD').all()
tokens.head()

,id,symbol,base,quote,active
0,0GUSD,0G/USD,0G,USD,True
1,1INCHUSD,1INCH/USD,1INCH,USD,True
2,2ZUSD,2Z/USD,2Z,USD,True
3,AAVEUSD,AAVE/USD,AAVE,USD,True
4,ABUSD,AB/USD,AB,USD,True


In [0]:
#| echo: false
#| output: asis
show_doc(save_file)

---

[source](https://github.com/silvaac/token_data/blob/main/token_data/coinbase.py#L175){target="_blank" style="float:right; font-size:smaller"}

### save_file

```python
def save_file(
    df, folder_path, file_name, type:str='parquet'
):
```

*Save a pandas DataFrame to a file in either CSV or Parquet format.*

Args:
    df (pandas.DataFrame): The DataFrame to save
    folder_path (str): Directory path where the file will be saved
    file_name (str): Name of the file without extension
    type (str, optional): File format - either "csv" or "parquet". Defaults to "parquet"

The function saves the DataFrame to the specified path, handling the file extension automatically.
For CSV files, the index is not saved. Creates the folder if it doesn't exist.

In [0]:
#| echo: false
#| output: asis
show_doc(file_name_to_symbol)

---

### file_name_to_symbol

```python
def file_name_to_symbol(
    file_name
):
```

*Convert a file name back into a ccxt symbol.*

Example: 'BTC-USD_1h.parquet' -> 'BTC/USD'

In [0]:
#| echo: false
#| output: asis
show_doc(symbol_to_file_name)

---

### symbol_to_file_name

```python
def symbol_to_file_name(
    symbol, timeframe:str='1h'
):
```

*Convert a ccxt symbol and timeframe into a file name (without extension).*

Example: ('BTC/USD', '1h') -> 'BTC-USD_1h'

In [ ]:
#|eval: false
# Offline test: save_file round-trip and file-name helpers
tmp_dir = tempfile.mkdtemp()
save_file(df_sample, tmp_dir, 'TEST-USD_1h', type='parquet')
df_back = pd.read_parquet(f"{tmp_dir}/TEST-USD_1h.parquet")
assert df_back.shape == df_sample.shape
assert list(df_back.columns) == list(df_sample.columns)
assert symbol_to_file_name('BTC/USD', '1h') == 'BTC-USD_1h'
assert file_name_to_symbol('BTC-USD_1h.parquet') == 'BTC/USD'
print('save_file round-trip OK')

save_file round-trip OK


In [0]:
#| echo: false
#| output: asis
show_doc(kraken_to_file)

---

### kraken_to_file

```python
def kraken_to_file(
    folder_path:str='../data/kraken', token_list:list=['BTC/USD', 'ETH/USD'], type:str='parquet', timeframe:str='1h',
    refresh_hours:int=24, all_tokens:bool=True, pause:int=1, verbose:bool=False
):
```

*Downloads and maintains historical Kraken OHLCV data, saving one file per pair.*

Args:
    folder_path (str): Path where pair data files will be stored, named after the
        exchange. Defaults to "../data/kraken"
    token_list (list): List of ccxt symbols to process. Defaults to ['BTC/USD', 'ETH/USD']
    type (str): File format to save data - either "csv" or "parquet". Defaults to "parquet"
    timeframe (str): Candle timeframe (e.g. "1m", "1h", "1d"). Defaults to "1h"
    refresh_hours (int): Hours of the most recent data to re-download when updating.
        Defaults to 24
    all_tokens (bool): If True, includes any additional pairs found in the folder path.
        Defaults to True
    pause (int): Seconds to wait between pairs. Defaults to 1
    verbose (bool): If True, prints download progress. Defaults to False

The function:
- Creates the folder_path if it doesn't exist
- Date/Time is UTC
- For each pair, checks if a data file exists:
    - If exists: Loads the file and appends new data, refreshing the last `refresh_hours`
    - If not exists: Downloads the maximum available history (~720 candles on Kraken)
- Saves data in the specified format, handling duplicates and sorting by datetime

#### Example / tests

In [ ]:
#|eval: false
# Live test: download BTC/USD into a temporary exchange folder, then re-run incrementally
import tempfile
kraken_dir = tempfile.mkdtemp()
kraken_to_file(folder_path=kraken_dir, token_list=['BTC/USD'], type='parquet', timeframe='1h', pause=0)
assert os.path.exists(f"{kraken_dir}/BTC-USD_1h.parquet")
df1 = pd.read_parquet(f"{kraken_dir}/BTC-USD_1h.parquet")
n1 = len(df1)
assert n1 > 0
# Incremental re-run must not shrink the file and must not create duplicates
kraken_to_file(folder_path=kraken_dir, token_list=['BTC/USD'], type='parquet', timeframe='1h', pause=0)
df2 = pd.read_parquet(f"{kraken_dir}/BTC-USD_1h.parquet")
assert len(df2) >= n1
assert not df2['datetime'].duplicated().any()
print(f"First run: {n1} rows, second run: {len(df2)} rows")

Processing BTC/USD
Processing BTC/USD
First run: 700 rows, second run: 700 rows


In [17]:
#|eval: false
# Download / update a set of USD pairs into the exchange-named data folder
# For all USD pairs use: token_list = kraken_usd_tokens()['symbol'].tolist()
kraken_to_file(folder_path="../data/kraken",
               token_list=['BTC/USD', 'ETH/USD', 'SOL/USD'],
               type="parquet", timeframe='1h')

Processing SOL/USD
Processing BTC/USD
Processing ETH/USD
